<a href="https://colab.research.google.com/github/Rawan-Almasri/text-obfuscation-ner-pipeline/blob/main/text_obfuscation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import os
os.environ["JUPYTER_WIDGETS_DISABLE"] = "1"

In [2]:
from google.colab import drive
import os
!pip install python-docx
from docx import Document

In [3]:
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [4]:
def get_file_path():
  # file_name = input ()
  file_name = "text_example.docx"
  file_path  = f"/content/drive/MyDrive/{file_name}"
  return file_path

In [5]:
def read_file(file_path):

    # Check if file exists
    if not os.path.exists(file_path):
        return "File not found!"

    # Read text file
    if file_path.endswith(".txt"):
        with open(file_path, "r", encoding="utf-8") as f:
            content = f.read()
        return content

    # Read word file
    elif file_path.endswith(".docx"):
        doc = Document(file_path)
        content = "\n".join([para.text for para in doc.paragraphs])
        return content

    else:
        return "Unsupported file type. Please provide a .txt or .docx file."

In [6]:
def get_enities():
  arr = input("Enter categories of entities to anonymize: ").split()
  # print("The entities are:", arr)
  return arr


In [7]:
!pip install gliner==0.1.12

In [8]:
from gliner import GLiNER
model = GLiNER.from_pretrained("numind/NuNerZero")

def merge_entities(entities, text):
    if not entities:
        return []
    merged = []
    current = entities[0].copy()   # copy to avoid mutating original dicts
    for next_entity in entities[1:]:
        if next_entity['label'] == current['label'] and (next_entity['start'] == current['end'] or next_entity['start'] == current['end'] + 1):
            current['text'] = text[current['start']: next_entity['end']].strip()
            current['end'] = next_entity['end']
        else:
            merged.append(current)
            current = next_entity.copy()
    merged.append(current)
    return merged


pytorch_model.bin:   0%|          | 0.00/1.80G [00:00<?, ?B/s]

gliner_config.json:   0%|          | 0.00/634 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/580 [00:00<?, ?B/s]

spm.model:   0%|          | 0.00/2.46M [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/874M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/874M [00:00<?, ?B/s]

In [9]:
def use_model(labels,text):
# NuZero requires labels to be lower-cased!
  labels = [l.lower() for l in labels]
  entities = model.predict_entities(text, labels)
  entities = merge_entities(entities, text)   # <-- pass text here
  # entities = merge_entities(entities)
  for entity in entities:
    print(entity["text"], "=>", entity["label"], "-" ,entity ['score'])
  return entities


In [10]:
def anonymize_text(text, entities):
    if not entities:
        return text

    entities = sorted(entities, key=lambda e: e['start'])

    anonymized = []
    last_idx = 0

    for ent in entities:
        anonymized.append(text[last_idx:ent['start']])
        anonymized.append("****")
        last_idx = ent['end']

    anonymized.append(text[last_idx:])

    return "".join(anonymized)


In [11]:
def crete_file_to_save(anonymized_text, origin_file_path):
  if origin_file_path.endswith(".txt"):
    file_path = "/content/drive/MyDrive/anonymized_text_file.txt"
    with open(file_path, "w", encoding="utf-8") as f:
       f.write(anonymized_text)
    print(f"TXT file saved at: {file_path}")
  else :
    doc = Document()
    doc.add_heading("Anonymized_text_file", 0)
    doc.add_paragraph(anonymized_text)
    file_path = "/content/drive/MyDrive/anonymized_text_file.docx"
    doc.save(file_path)
    print(f"Word file saved at: {file_path}")



In [12]:
#pipeline
file = get_file_path()
file_content = read_file (file)
print (f"The content is {file_content}")
enities = get_enities()
# the input is:
#person organization location country
result_entities  = use_model(enities,file_content)
anonymized_text  = anonymize_text(file_content, result_entities)
print(anonymized_text)
crete_file_to_save (anonymized_text , file)

The content is In 2022, Elon Musk visited Berlin, Germany, to meet with executives from Tesla and discuss the expansion of the Gigafactory in Europe. 
During the same trip, he also gave a talk at the Technical University of Berlin about the future of electric vehicles and sustainable energy. 
Later that year, Musk attended a conference in San Francisco, where he met with representatives from Google and Microsoft to explore opportunities in artificial intelligence. 
Meanwhile, Angela Merkel, the former Chancellor of Germany, praised the collaboration between European countries and the United States in advancing green technologies. 
At the same time, the United Nations organized a summit in New York City, gathering leaders from organizations like the World Health Organization and UNICEF to address global health challenges.
Enter categories of entities to anonymize: person organization location country
Elon Musk => person - 0.999902606010437
Berlin => location - 0.9833258390426636
Germany